# Row Group Pruning in Real Parquet Files

Companion notebook for the [Columnar Formats: Parquet and ORC](https://codemia.io/courses/big_data_engines/bd_columnar_formats_parquet_and_orc) lesson on Codemia.

**What you will build**: three real Parquet files holding the same two million rows, written three ways. You will read their
footers, work out which row groups a query can skip, and measure the bytes and the seconds each
layout costs. The same query reads two percent of one file and all of another.

## Setup

In [ ]:
%pip install -q pyarrow numpy matplotlib

In [ ]:
import os
import time
import tempfile

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
WORK = tempfile.mkdtemp(prefix='parquet-pruning-')
print('pyarrow', pa.__version__)
print('writing files under', WORK)

## Concept recap

A Parquet file is a sequence of row groups, and each row group stores each column separately with a
small statistics block: the minimum, the maximum and the null count. A reader that knows your
predicate can compare it against those statistics and decide not to open a column chunk at all.

That is the whole mechanism, and it has one consequence that decides how your table should be
written: statistics prune a row group only when the values in it are clustered. A column whose
values are scattered uniformly has a minimum near the global minimum and a maximum near the global
maximum in every single row group, and nothing can ever be skipped.

## Two million rows, written three ways

In [ ]:
N_ROWS = 1_000_000
DAYS = 30
DAY = 86_400
START = 1_760_000_000              # an arbitrary wall clock origin
ROW_GROUP = 50_000

event_time = START + rng.integers(0, DAYS * DAY, size=N_ROWS)
user_id = rng.integers(0, 400_000, size=N_ROWS)
country = rng.choice(np.array(['US', 'GB', 'DE', 'IN', 'BR', 'JP', 'CA', 'AU']),
                     size=N_ROWS, p=[.34, .12, .11, .17, .09, .08, .05, .04])
amount_cents = rng.integers(99, 50_000, size=N_ROWS)

# An opaque 32 byte blob per row, incompressible on purpose. Most real tables carry
# something like this, and it is most of the bytes you are trying not to read.
payload = np.frombuffer(rng.integers(0, 256, size=N_ROWS * 32, dtype=np.uint8).tobytes(),
                        dtype='S32')

table = pa.table({
    'event_time': event_time,
    'user_id': user_id,
    'country': pa.array(country).cast(pa.dictionary(pa.int32(), pa.string())),
    'amount_cents': amount_cents,
    'payload': payload,
})

sorted_table = table.sort_by('event_time')

LAYOUTS = {}
LAYOUTS['unsorted'] = os.path.join(WORK, 'unsorted.parquet')
LAYOUTS['sorted_by_time'] = os.path.join(WORK, 'sorted.parquet')
LAYOUTS['one_row_group'] = os.path.join(WORK, 'one_group.parquet')

pq.write_table(table, LAYOUTS['unsorted'], row_group_size=ROW_GROUP, compression='snappy')
pq.write_table(sorted_table, LAYOUTS['sorted_by_time'], row_group_size=ROW_GROUP, compression='snappy')
pq.write_table(sorted_table, LAYOUTS['one_row_group'], row_group_size=N_ROWS, compression='snappy')

for name, path in LAYOUTS.items():
    meta = pq.ParquetFile(path).metadata
    print(f'{name:>16}  {meta.num_row_groups:>4} row groups  '
          f'{os.path.getsize(path) / 2 ** 20:>7.1f} MB on disk')

Identical rows, identical schema, identical compression. The only differences are the order the
rows were written in and the size of a row group, and neither of those is usually treated as a
decision. The rest of the notebook is about how much they decide.

## Reading the footer yourself

In [ ]:
def column_index(meta, name):
    return [meta.schema.column(i).name for i in range(meta.num_columns)].index(name)


def row_group_ranges(path, column='event_time'):
    """Pull the min and max of one column out of every row group's statistics block."""
    meta = pq.ParquetFile(path).metadata
    j = column_index(meta, column)
    out = []
    for i in range(meta.num_row_groups):
        stats = meta.row_group(i).column(j).statistics
        out.append((stats.min, stats.max, meta.row_group(i).num_rows))
    return out


for name in ['unsorted', 'sorted_by_time']:
    ranges = row_group_ranges(LAYOUTS[name])
    print(name)
    for i in [0, 1, 2]:
        lo, hi, rows = ranges[i]
        print(f'  row group {i}: {rows:,} rows, event_time spans '
              f'{(hi - lo) / DAY:>6.2f} days')
    spans = [(hi - lo) / DAY for lo, hi, _ in ranges]
    print(f'  mean span across all {len(ranges)} row groups: {np.mean(spans):.2f} days')
    print()

There is the difference, in one number. In the sorted file a row group covers a fraction of a day,
so a query for one day can rule out almost every group by arithmetic on two integers. In the
unsorted file every row group covers the entire month, because a random sample of two million rows
taken a hundred thousand at a time contains the oldest and the newest row every time.

## What a one day query actually reads

In [ ]:
PROJECTED = ['event_time', 'amount_cents']       # the columns the query needs
LO = START + 7 * DAY                             # a single day, a week into the month
HI = LO + DAY


def pruning_report(path, lo, hi, column='event_time', projected=PROJECTED):
    """Skip a row group when its statistics prove it holds no value in [lo, hi)."""
    meta = pq.ParquetFile(path).metadata
    cols = [column_index(meta, c) for c in projected]
    j = column_index(meta, column)

    touched = rows = read_bytes = 0
    for i in range(meta.num_row_groups):
        stats = meta.row_group(i).column(j).statistics
        if stats.max < lo or stats.min >= hi:        # cannot contain a matching row
            continue
        touched += 1
        rows += meta.row_group(i).num_rows
        read_bytes += sum(meta.row_group(i).column(c).total_compressed_size for c in cols)

    full = sum(meta.row_group(i).column(c).total_compressed_size
               for i in range(meta.num_row_groups) for c in cols)
    return meta.num_row_groups, touched, rows, read_bytes, full


print(f"{'layout':>16}{'groups':>9}{'touched':>9}{'rows scanned':>15}{'MB read':>10}{'of file':>9}")
for name, path in LAYOUTS.items():
    groups, touched, rows, read_bytes, full = pruning_report(path, LO, HI)
    file_mb = os.path.getsize(path) / 2 ** 20
    print(f'{name:>16}{groups:>9}{touched:>9}{rows:>15,}'
          f'{read_bytes / 2 ** 20:>10.2f}{read_bytes / 2 ** 20 / file_mb:>8.1%}')

The query is the same query in all three rows. One layout reads a couple of percent of the file,
one reads every byte of the two columns it projected, and one cannot skip anything because there is
nothing to skip: a single row group is an all or nothing decision.

Note what the projection alone bought you, which is the gap between the MB read column and the file
size. Even in the worst layout, naming two columns out of five avoided most of the file, because the
payload column is stored separately. Column pruning is the cheaper of the two mechanisms and the one
you get for free by not writing `SELECT *`.

## The same experiment, measured instead of calculated

In [ ]:
def timed_read(path, lo, hi, repeats=3):
    best = float('inf')
    rows = 0
    for _ in range(repeats):
        start = time.perf_counter()
        t = pq.read_table(path, columns=PROJECTED,
                          filters=[('event_time', '>=', lo), ('event_time', '<', hi)])
        best = min(best, time.perf_counter() - start)
        rows = t.num_rows
    return best, rows


print(f"{'layout':>16}{'rows returned':>16}{'seconds':>10}")
for name, path in LAYOUTS.items():
    seconds, rows = timed_read(path, LO, HI)
    print(f'{name:>16}{rows:>16,}{seconds:>10.3f}')

Every layout returns the same rows, because pruning is an optimisation and not a filter. The
seconds are the point, and on a local SSD they understate the difference badly: on object storage
each surviving row group is a separate ranged GET with its own round trip, so the gap between the
layouts widens from a factor of a few to a factor of tens.

## Row group size is a trade, not a setting to maximise

In [ ]:
sizes = [5_000, 25_000, 50_000, 250_000, N_ROWS]
measured = []

for size in sizes:
    path = os.path.join(WORK, f'rg_{size}.parquet')
    pq.write_table(sorted_table, path, row_group_size=size, compression='snappy')
    groups, touched, rows, read_bytes, _ = pruning_report(path, LO, HI)
    measured.append((size, groups, touched, rows, read_bytes / 2 ** 20,
                     os.path.getsize(path) / 2 ** 20))

print(f"{'row group':>12}{'groups':>9}{'touched':>9}{'rows scanned':>15}{'MB read':>10}{'file MB':>10}")
for size, groups, touched, rows, mb, file_mb in measured:
    print(f'{size:>12,}{groups:>9}{touched:>9}{rows:>15,}{mb:>10.2f}{file_mb:>10.1f}')

In [ ]:
labels = [f'{s // 1000}k' for s, *_ in measured]
scanned = [rows for _, _, _, rows, _, _ in measured]
footprint = [file_mb for *_, file_mb in measured]

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.bar(labels, scanned, color='tab:blue', alpha=0.8)
ax1.set_ylabel('rows scanned for a one day query', color='tab:blue')
ax1.set_xlabel('rows per row group')
ax1.axhline(N_ROWS / DAYS, color='black', linestyle='--',
            label=f'rows that actually match ({int(N_ROWS / DAYS):,})')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(labels, footprint, color='tab:red', marker='o')
ax2.set_ylabel('file size on disk, MB', color='tab:red')

plt.title('Small row groups prune better and compress worse')
plt.show()

Smaller row groups prune closer to the rows you asked for, and they cost you on the other side: more
footer metadata, shorter runs for the encoders, more requests to fetch the same data. The usual
advice of a hundred thousand to a million rows, or a target of a hundred and twenty eight megabytes,
is the middle of this curve rather than a magic number.

## The predicate you did not cluster for

In [ ]:
user_ranges = row_group_ranges(LAYOUTS['sorted_by_time'], column='user_id')
spans = [(hi - lo) for lo, hi, _ in user_ranges]
print(f'user_id spans the full range in {sum(1 for s in spans if s > 399_000)} of '
      f'{len(spans)} row groups')

target = int(user_id[0])
groups, touched, rows, read_bytes, _ = pruning_report(
    LAYOUTS['sorted_by_time'], target, target + 1, column='user_id')
print()
print('a point lookup on user_id against the time-sorted file')
print(f'  row groups touched {touched} of {groups}')
print(f'  rows scanned       {rows:,} to find a handful')

by_user = table.sort_by('user_id')
path = os.path.join(WORK, 'sorted_by_user.parquet')
pq.write_table(by_user, path, row_group_size=ROW_GROUP, compression='snappy')

groups, touched, rows, _, _ = pruning_report(path, target, target + 1, column='user_id')
print()
print('the same lookup against a user-sorted copy')
print(f'  row groups touched {touched} of {groups}')
print(f'  rows scanned       {rows:,}')

groups, touched, rows, _, _ = pruning_report(path, LO, HI)
print()
print('and the one day query against that same user-sorted copy')
print(f'  row groups touched {touched} of {groups}')
print(f'  rows scanned       {rows:,}')

This is the constraint that makes table layout a design problem rather than a checkbox. There is one
physical order. Sorting by time makes time queries cheap and leaves user lookups scanning the whole
table; sorting by user does the reverse, exactly and symmetrically.

The ways out all cost something. A composite sort on `(country, event_time)` prunes well for queries
that filter on both and poorly for queries that filter only on the second. A space filling curve,
which is what Delta's `ZORDER` and Iceberg's `sort_order` with multiple columns approximate, spreads
the benefit across several columns and gives none of them the clustering a single sort would. A
secondary index, or Parquet's own page level bloom filters, answers point lookups without changing
the order at all, at the cost of writing and maintaining it. Pick by reading the queries, not by
reading the documentation.

## What to take away

- Statistics prune row groups. Clustering is what makes statistics selective, so the write order is
  the query plan's most important input.
- Projection and predicate pushdown are different mechanisms. The first is free and you are probably
  wasting it with `SELECT *`; the second has to be designed for.
- A single enormous row group is the layout with the best compression and no pruning at all.
- Min and max on a high cardinality unsorted column are noise. Measure the span of your row groups
  before you believe a pruning story.
- One table has one order. Every multi predicate workload is a compromise between orders, and saying
  which queries you are optimising for is the actual decision.